# NB2 — Pondération, n-grams de caractères et représentations hybrides

Notebook des pipelines **P06 à P10**.

## Portée du notebook

Ce notebook étudie la **robustesse aux tweets courts et bruités** :
- pondération sous-linéaire (`sublinear_tf`) ;
- n-grams de caractères ;
- combinaison **mots + caractères**.

Comme pour NB1, les jeux `train.csv` et `test.csv` sont supposés déjà présents dans `../../data/`.

In [9]:
# Pour un run sur Colab

''' 
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/Disaster-Tweets-NLP"
MODELS_DIR = f"{PROJECT_ROOT}/notebooks/models_training"

%cd "{MODELS_DIR}"

import sys
if MODELS_DIR not in sys.path:
    sys.path.append(MODELS_DIR)

print("Projet :", PROJECT_ROOT)
print("Dossier courant :", MODELS_DIR)

'''

' \nfrom google.colab import drive\ndrive.mount(\'/content/drive\')\n\nPROJECT_ROOT = "/content/drive/MyDrive/Disaster-Tweets-NLP"\nMODELS_DIR = f"{PROJECT_ROOT}/notebooks/models_training"\n\n%cd "{MODELS_DIR}"\n\nimport sys\nif MODELS_DIR not in sys.path:\n    sys.path.append(MODELS_DIR)\n\nprint("Projet :", PROJECT_ROOT)\nprint("Dossier courant :", MODELS_DIR)\n\n'

In [10]:
# Installation éventuelle (décommente si nécessaire)
# !pip install pandas numpy scikit-learn scipy matplotlib gensim sentence-transformers openpyxl mlflow

from collections import OrderedDict

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import Normalizer

from nlp_disaster_utils import (
    seed_everything,
    load_train_test_xy,
    evaluate_sklearn_pipeline,
    round_results,
    metric_matrix_from_results,
    save_results_bundle,
    GensimMeanEmbeddingVectorizer,
    SentenceTransformerVectorizer,
)

seed_everything(42)
import mlflow


from mlflow_utils import (
    setup_mlflow_tracking,
    fit_evaluate_and_log_sklearn_pipeline,
)


In [11]:
DATA_DIR = "../../data/processed_data"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TEST_PATH = f"{DATA_DIR}/test.csv"

TEXT_COL = "text"
LABEL_COL = "target"
USE_AUX_TEXT_COLUMNS = False
LOWERCASE_TEXT = False
RANDOM_STATE = 42

OUTPUT_STEM = "NB2_weighting_char_hybrid"
RESULTS_DIR = "../../outputs/NB2"

# Configuration MLflow
from pathlib import Path
MLFLOW_EXPERIMENT_NAME = "DT_NB2_weighting_char_hybrid"
MLFLOW_TRACKING_URI = Path("../../outputs/mlruns").resolve().as_uri()
MLFLOW_LOG_MODEL = False

tracking_uri = setup_mlflow_tracking(
    experiment_name=MLFLOW_EXPERIMENT_NAME,
    tracking_uri=MLFLOW_TRACKING_URI,
)
print("MLflow tracking URI :", tracking_uri)
print("MLflow experiment   :", MLFLOW_EXPERIMENT_NAME)


MLflow tracking URI : file:///C:/Users/DELL/Documents/Classes/ISE2/ISE2_2026/SEM2/ML2/Projet/Disaster-Tweets-NLP/outputs/mlruns
MLflow experiment   : DT_NB2_weighting_char_hybrid


In [12]:
df_train, X_train, y_train, df_test, X_test, y_test = load_train_test_xy(
    train_path=TRAIN_PATH,
    test_path=TEST_PATH,
    text_col=TEXT_COL,
    label_col=LABEL_COL,
    use_extra_cols=USE_AUX_TEXT_COLUMNS,
    lowercase=LOWERCASE_TEXT,
)

print("Taille train :", len(X_train))
print("Taille test  :", len(X_test))
print("\nDistribution des classes - train :")
print(y_train.value_counts(normalize=True).sort_index())
print("\nDistribution des classes - test :")
print(y_test.value_counts(normalize=True).sort_index())

Taille train : 9096
Taille test  : 2274

Distribution des classes - train :
target
0    0.814094
1    0.185906
Name: proportion, dtype: float64

Distribution des classes - test :
target
0    0.813984
1    0.186016
Name: proportion, dtype: float64


In [13]:
word_tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
)
char_tfidf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
)

pipelines = OrderedDict({
    "P06_TFIDF_Sublinear_LogReg": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)),
        ("clf", LogisticRegression(max_iter=2500, C=1.0)),
    ]),
    "P07_CharTFIDF_LogReg": Pipeline([
        ("vect", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2)),
        ("clf", LogisticRegression(max_iter=2500, C=1.0)),
    ]),
    "P08_CharTFIDF_LinearSVC": Pipeline([
        ("vect", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2)),
        ("clf", LinearSVC(C=1.0)),
    ]),
    "P09_WordChar_Hybrid_LogReg": Pipeline([
        ("features", FeatureUnion([
            ("word_tfidf", word_tfidf),
            ("char_tfidf", char_tfidf),
        ])),
        ("clf", LogisticRegression(max_iter=3000, C=1.0)),
    ]),
    "P10_WordChar_Hybrid_LinearSVC": Pipeline([
        ("features", FeatureUnion([
            ("word_tfidf", word_tfidf),
            ("char_tfidf", char_tfidf),
        ])),
        ("clf", LinearSVC(C=1.0)),
    ]),
})

In [14]:
resultats = []

for nom_pipeline, pipeline in pipelines.items():
    print(f"Entraînement -> {nom_pipeline}")
    display(pipeline)
    print("-" * 80)

    metrics = fit_evaluate_and_log_sklearn_pipeline(
        name=nom_pipeline,
        estimator=pipeline,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        notebook_name="NB2",
        family_name="weighting_char_hybrid",
        output_dir=RESULTS_DIR,
        log_model=MLFLOW_LOG_MODEL,
    )
    resultats.append(metrics)

results_df = round_results(pd.DataFrame(resultats))
display(results_df)


Entraînement -> P06_TFIDF_Sublinear_LogReg


Pipeline(steps=[('vect',
                 TfidfVectorizer(max_df=0.95, min_df=2, ngram_range=(1, 2),
                                 sublinear_tf=True)),
                ('clf', LogisticRegression(max_iter=2500))])

--------------------------------------------------------------------------------
Entraînement -> P07_CharTFIDF_LogReg


Pipeline(steps=[('vect',
                 TfidfVectorizer(analyzer='char_wb', min_df=2,
                                 ngram_range=(3, 5))),
                ('clf', LogisticRegression(max_iter=2500))])

--------------------------------------------------------------------------------
Entraînement -> P08_CharTFIDF_LinearSVC


...

Pipeline(steps=[('features',
                 FeatureUnion(transformer_list=[('word_tfidf',
                                                 TfidfVectorizer(max_df=0.95,
                                                                 min_df=2,
                                                                 ngram_range=(1,
                                                                              2))),
                                                ('char_tfidf',
                                                 TfidfVectorizer(analyzer='char_wb',
                                                                 min_df=2,
                                                                 ngram_range=(3,
                                                                              5)))])),
                ('clf', LogisticRegression(max_iter=3000))])

--------------------------------------------------------------------------------
Entraînement -> P10_WordChar_Hybrid_LinearSVC


Pipeline(steps=[('features',
                 FeatureUnion(transformer_list=[('word_tfidf',
                                                 TfidfVectorizer(max_df=0.95,
                                                                 min_df=2,
                                                                 ngram_range=(1,
                                                                              2))),
                                                ('char_tfidf',
                                                 TfidfVectorizer(analyzer='char_wb',
                                                                 min_df=2,
                                                                 ngram_range=(3,
                                                                              5)))])),
                ('clf', LinearSVC())])

--------------------------------------------------------------------------------


pipeline,P06_TFIDF_Sublinear_LogReg,P07_CharTFIDF_LogReg,P08_CharTFIDF_LinearSVC,P09_WordChar_Hybrid_LogReg,P10_WordChar_Hybrid_LinearSVC
train_accuracy,0.8989,0.9070,0.9919,0.9394,0.9989
train_precision_macro,0.9372,0.9341,0.9929,0.9594,0.9986
train_recall_macro,0.7309,0.7565,0.9802,0.8407,0.9977
train_f1_macro,0.7858,0.8098,0.9864,0.8858,0.9982
train_precision_weighted,0.9080,0.9128,0.9919,0.9425,0.9989
train_recall_weighted,0.8989,0.9070,0.9919,0.9394,0.9989
train_f1_weighted,0.8836,0.8952,0.9918,0.9350,0.9989
train_precision_class_0,0.8907,0.9003,0.9913,0.9325,0.9991
train_recall_class_0,0.9982,0.9961,0.9988,0.9978,0.9996
train_f1_class_0,0.9414,0.9458,0.9950,0.9641,0.9993


In [15]:
save_results_bundle(pd.DataFrame(resultats), output_dir=RESULTS_DIR, stem=OUTPUT_STEM)
print(f"Fichiers CSV/XLSX enregistrés dans {RESULTS_DIR}")

Fichiers CSV/XLSX enregistrés dans ../../outputs/NB2
